In [4]:
import pandas as pd

# Load and preprocess the MPRA data
df_mpra = pd.read_csv(
    '/media/zihengc/T7/mpra3_lib_analysis/indexing/MPRA3_Contributor_20231108_unique_GeneName_BarcodeEnhancerPair.csv',
    index_col=0
)
df_mpra = df_mpra.drop_duplicates(subset='enhancer_id', keep='first')

# Function to pad sequences to 500bp
def pad_sequence(seq):
    if len(seq) < 500:
        total_padding = 500 - len(seq)
        left_padding = 'N' * (total_padding // 2)
        right_padding = 'N' * (total_padding - len(left_padding))
        return f"{left_padding}{seq}{right_padding}"
    else:
        return seq

# Function to save the DataFrame as a FASTA file
def save_as_fasta(df, file_name):
    with open(file_name, 'w') as f:
        for _, row in df.iterrows():
            f.write(f">{row['enhancer_id']}\n{row['padded_seq']}\n")

# Function to process data based on FDR condition
def process_data(fdr_condition, file_suffix):
    df_differential = pd.read_csv(
        '/media/zihengc/T7/mpra3_lib_analysis/enhancer_activities_differential/20240626_comparative_THP1_LPSIFNGvsNaive_DifferentialEnhancer.csv',
        index_col=0
    )
    df_filtered = df_differential[fdr_condition(df_differential['fdr'])]
    df_mpra_filtered = df_mpra[df_mpra['enhancer_id'].isin(df_filtered.index)].copy()
    df_mpra_filtered['padded_seq'] = df_mpra_filtered['enhancer_seq'].apply(pad_sequence)
    
    # Save as FASTA
    fasta_file = f'sequence_for_prediction/{file_suffix}.fasta'
    save_as_fasta(df_mpra_filtered, fasta_file)
    
    # Reset index and save as CSV
    csv_file = f'sequence_for_prediction/{file_suffix}.csv'
    df_mpra_filtered.reset_index().to_csv(csv_file)

# Define conditions and file suffixes
conditions = [
    (lambda fdr: fdr <= 0.05, '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1LPSIFNG_differentialMPRAenhancers'),
    (lambda fdr: fdr >= 0.2, '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1LPSIFNG_differentialMPRAenhancers_fdr02background')
]

# Process data for each condition
for fdr_condition, file_suffix in conditions:
    process_data(fdr_condition, file_suffix)


/tmp/ipykernel_112919/985728599.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_500['padded_seq'] = df_500['enhancer_seq'].apply(pad_sequence)


In [2]:
import pandas as pd

# Load and preprocess the MPRA data
df_mpra = pd.read_csv(
    '/media/zihengc/T7/mpra3_lib_analysis/indexing/MPRA3_Contributor_20231108_unique_GeneName_BarcodeEnhancerPair.csv',
    index_col=0
)
df_mpra = df_mpra.drop_duplicates(subset='enhancer_id', keep='first')

# Function to pad sequences to 500bp
def pad_sequence(seq):
    if len(seq) < 500:
        total_padding = 500 - len(seq)
        left_padding = 'N' * (total_padding // 2)
        right_padding = 'N' * (total_padding - len(left_padding))
        return f"{left_padding}{seq}{right_padding}"
    else:
        return seq

# Function to save the DataFrame as a FASTA file
def save_as_fasta(df, file_name):
    with open(file_name, 'w') as f:
        for _, row in df.iterrows():
            f.write(f">{row['enhancer_id']}\n{row['padded_seq']}\n")

# Function to process data based on p-value condition
def process_data(mad_file, pval_condition, file_suffix, save_csv=False):
    df_differential = pd.read_csv(mad_file,index_col=0)
    df_filtered = df_differential[pval_condition(df_differential['pval.mad'])]
    df_mpra_filtered = df_mpra[df_mpra['enhancer_id'].isin(df_filtered.index)].copy()
    df_mpra_filtered['padded_seq'] = df_mpra_filtered['enhancer_seq'].apply(pad_sequence)
    
    # Save as FASTA
    fasta_file = f'sequence_for_prediction/{file_suffix}.fasta'
    save_as_fasta(df_mpra_filtered, fasta_file)
    
    # Optionally save as CSV
    if save_csv:
        csv_file = f'sequence_for_prediction/{file_suffix}.csv'
        df_mpra_filtered.reset_index().to_csv(csv_file, index=False)


In [17]:
# Define conditions, file suffixes, and whether to save CSV
conditions = [
    (   lambda pval: pval <= 0.05,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_MouseBrain_activeMPRAenhancers',
        True # Change to True if you want to save the CSV
    ),
    (   lambda pval: pval >= 0.2,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_MouseBrain_activeMPRAenhancers_fdr02background',
        True)]
# Process data for each condition
for pval_condition, file_suffix, save_csv in conditions:
    process_data('/media/zihengc/T7/mpra3_lib_analysis/enhancer_activities/MAD_OneTail_NoControl/20241126_MAD_Brain.csv',pval_condition, file_suffix, save_csv)


    # Define conditions, file suffixes, and whether to save CSV
conditions = [
    (   lambda pval: pval <= 0.05,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1Macrophage_activeMPRAenhancers',
        True  # Change to True if you want to save the CSV
    ),
    (   lambda pval: pval >= 0.2,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1Macrophage_activeMPRAenhancers_fdr02background',
        True
    )]
# Process data for each condition
for pval_condition, file_suffix, save_csv in conditions:
    process_data('/media/zihengc/T7/mpra3_lib_analysis/enhancer_activities/MAD_OneTail_NoControl/20241126_MAD_THP1Macrophage.csv',pval_condition, file_suffix, save_csv)


In [ ]:
# Define conditions, file suffixes, and whether to save CSV
conditions = [
    (   lambda pval: pval <= 0.05,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1Naive_activeMPRAenhancers',
        True  # Change to True if you want to save the CSV
    ),
    (   lambda pval: pval >= 0.2,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1Naive_activeMPRAenhancers_fdr02background',
        True
    )]
# Process data for each condition
for pval_condition, file_suffix, save_csv in conditions:
    process_data('/media/zihengc/T7/mpra3_lib_analysis/enhancer_activities/MAD_OneTail_NoControl/20240824_MAD_THP1_Naive.csv',pval_condition, file_suffix, save_csv)

# Define conditions, file suffixes, and whether to save CSV
conditions = [
    (   lambda pval: pval <= 0.05,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1IFNB_activeMPRAenhancers',
        True  # Change to True if you want to save the CSV
    ),
    (   lambda pval: pval >= 0.2,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1IFNB_activeMPRAenhancers_fdr02background',
        True
    )]
# Process data for each condition
for pval_condition, file_suffix, save_csv in conditions:
    process_data('/media/zihengc/T7/mpra3_lib_analysis/enhancer_activities/MAD_OneTail_NoControl/20240824_MAD_THP1_IFNB.csv',pval_condition, file_suffix, save_csv)

# Define conditions, file suffixes, and whether to save CSV
conditions = [
    (   lambda pval: pval <= 0.05,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1IFNG_activeMPRAenhancers',
        True  # Change to True if you want to save the CSV
    ),
    (   lambda pval: pval >= 0.2,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1IFNG_activeMPRAenhancers_fdr02background',
        True
    )]
# Process data for each condition
for pval_condition, file_suffix, save_csv in conditions:
    process_data('/media/zihengc/T7/mpra3_lib_analysis/enhancer_activities/MAD_OneTail_NoControl/20240824_MAD_THP1_IFNG.csv',pval_condition, file_suffix, save_csv)

# Define conditions, file suffixes, and whether to save CSV
conditions = [
    (   lambda pval: pval <= 0.05,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1LPSIFNG_activeMPRAenhancers',
        True  # Change to True if you want to save the CSV
    ),
    (   lambda pval: pval >= 0.2,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1LPSIFNG_activeMPRAenhancers_fdr02background',
        True
    )]
# Process data for each condition
for pval_condition, file_suffix, save_csv in conditions:
    process_data('/media/zihengc/T7/mpra3_lib_analysis/enhancer_activities/MAD_OneTail_NoControl/20240824_MAD_THP1_LPSIFNG.csv',pval_condition, file_suffix, save_csv)


# Define conditions, file suffixes, and whether to save CSV
conditions = [
    (   lambda pval: pval <= 0.05,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_Cortex_activeMPRAenhancers',
        True  # Change to True if you want to save the CSV
    ),
    (   lambda pval: pval >= 0.2,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_Cortex_activeMPRAenhancers_fdr02background',
        True
    )]
# Process data for each condition
for pval_condition, file_suffix, save_csv in conditions:
    process_data('/media/zihengc/T7/mpra3_lib_analysis/enhancer_activities/MAD_OneTail_NoControl/20241126_MAD_Cortex.csv',pval_condition, file_suffix, save_csv)

# Define conditions, file suffixes, and whether to save CSV
conditions = [
    (   lambda pval: pval <= 0.05,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_Striatum_activeMPRAenhancers',
        True  # Change to True if you want to save the CSV
    ),
    (   lambda pval: pval >= 0.2,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_Striatum_activeMPRAenhancers_fdr02background',
        True
    )]
# Process data for each condition
for pval_condition, file_suffix, save_csv in conditions:
    process_data('/media/zihengc/T7/mpra3_lib_analysis/enhancer_activities/MAD_OneTail_NoControl/20241126_MAD_Striatum.csv',pval_condition, file_suffix, save_csv)


In [3]:
# Define conditions, file suffixes, and whether to save CSV
conditions = [
    (   lambda pval: pval <= 0.1,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1Naive_activeMPRAenhancers01',
        True  # Change to True if you want to save the CSV
    ),
    (   lambda pval: pval >= 0.2,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1Naive_activeMPRAenhancers_fdr02background',
        True
    )]
# Process data for each condition
for pval_condition, file_suffix, save_csv in conditions:
    process_data('/media/zihengc/T7/mpra3_lib_analysis/enhancer_activities/MAD_OneTail_NoControl/20240824_MAD_THP1_Naive.csv',pval_condition, file_suffix, save_csv)

# Define conditions, file suffixes, and whether to save CSV
conditions = [
    (   lambda pval: pval <= 0.1,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1IFNB_activeMPRAenhancers01',
        True  # Change to True if you want to save the CSV
    ),
    (   lambda pval: pval >= 0.2,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1IFNB_activeMPRAenhancers_fdr02background',
        True
    )]
# Process data for each condition
for pval_condition, file_suffix, save_csv in conditions:
    process_data('/media/zihengc/T7/mpra3_lib_analysis/enhancer_activities/MAD_OneTail_NoControl/20240824_MAD_THP1_IFNB.csv',pval_condition, file_suffix, save_csv)

# Define conditions, file suffixes, and whether to save CSV
conditions = [
    (   lambda pval: pval <= 0.1,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1IFNG_activeMPRAenhancers01',
        True  # Change to True if you want to save the CSV
    ),
    (   lambda pval: pval >= 0.2,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1IFNG_activeMPRAenhancers_fdr02background',
        True
    )]
# Process data for each condition
for pval_condition, file_suffix, save_csv in conditions:
    process_data('/media/zihengc/T7/mpra3_lib_analysis/enhancer_activities/MAD_OneTail_NoControl/20240824_MAD_THP1_IFNG.csv',pval_condition, file_suffix, save_csv)

# Define conditions, file suffixes, and whether to save CSV
conditions = [
    (   lambda pval: pval <= 0.1,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1LPSIFNG_activeMPRAenhancers01',
        True  # Change to True if you want to save the CSV
    ),
    (   lambda pval: pval >= 0.2,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_THP1LPSIFNG_activeMPRAenhancers_fdr02background',
        True
    )]
# Process data for each condition
for pval_condition, file_suffix, save_csv in conditions:
    process_data('/media/zihengc/T7/mpra3_lib_analysis/enhancer_activities/MAD_OneTail_NoControl/20240824_MAD_THP1_LPSIFNG.csv',pval_condition, file_suffix, save_csv)


# Define conditions, file suffixes, and whether to save CSV
conditions = [
    (   lambda pval: pval <= 0.1,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_Cortex_activeMPRAenhancers01',
        True  # Change to True if you want to save the CSV
    ),
    (   lambda pval: pval >= 0.2,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_Cortex_activeMPRAenhancers_fdr02background',
        True
    )]
# Process data for each condition
for pval_condition, file_suffix, save_csv in conditions:
    process_data('/media/zihengc/T7/mpra3_lib_analysis/enhancer_activities/MAD_OneTail_NoControl/20241126_MAD_Cortex.csv',pval_condition, file_suffix, save_csv)

# Define conditions, file suffixes, and whether to save CSV
conditions = [
    (   lambda pval: pval <= 0.1,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_Striatum_activeMPRAenhancers01',
        True  # Change to True if you want to save the CSV
    ),
    (   lambda pval: pval >= 0.2,
        '20241002_MPRA_500bp_SNPandControl_Ns_227bpenhancers_Striatum_activeMPRAenhancers_fdr02background',
        True
    )]
# Process data for each condition
for pval_condition, file_suffix, save_csv in conditions:
    process_data('/media/zihengc/T7/mpra3_lib_analysis/enhancer_activities/MAD_OneTail_NoControl/20241126_MAD_Striatum.csv',pval_condition, file_suffix, save_csv)
